# NYC analysis — exactly 70 groups, with offline or API travel times

**2,327 source census tracts → 70 analysis groups → a 70 × 70 travel matrix.**

The tracts are used only to aggregate population and race counts. They are **not** 2,327 routing locations. All demographic rows and travel matrix axes use the same 70 group IDs. `outputs/nyc_routes/groups_70.csv` is the combined 70-row table of coordinates, population and race counts.

The final analysis uses **WALK and TRANSIT generalized costs**, including fares, the shared local cost floor, destination population weighting, and race proportions. It uses saved MTA schedules and approximate walking, with an explicitly labeled transit proxy for missing scheduled coverage and can run to completion without any API. No Routes API requests have been made for this workflow. **Collection remains disabled until you approve it.** Running all cells leaves collection disabled.

This working copy uses the original notebook's demographic and PSSI definitions, with a reusable collector in `scripts/nyc_routes.py`. The byte-for-byte original is in `notebooks/reference/APMA4903_original.ipynb`. Use the project's `.venv` Python environment; no package installation or data downloads run in this notebook.

## Geographic design

The 2,327 NYC 2020 Census tracts are grouped into **70 custom analytical zones**, preserving whole tracts and borough boundaries. Borough zone counts are proportional to population using largest remainders: Bronx 12, Brooklyn 22, Manhattan 13, Queens 19, Staten Island 4. Spatial recursive bisection uses tract population and projected coordinates, never race composition.

Zone populations range from **119,233 to 135,465** (mean 125,774). All 8,804,190 residents and all 75 demographic counts are conserved. The routing point is the populated tract internal point nearest the zone's population-weighted mean of tract internal points. This is a tract-level approximation, not a household-weighted location or a verified transit stop. These are not official neighborhoods, and strict geographic contiguity is not guaranteed.

Rows are origins; columns are destinations. Directions are kept separate. There are 4,900 matrix entries, of which 70 diagonal entries use the local cost-floor model, leaving **4,830 requests per mode per departure time**. WALK + TRANSIT requires 9,660 requests. TRANSIT-only is the initial budget scenario; it does not reproduce the multimodal minimum in the original notebook.


In [ ]:
from pathlib import Path
import sys
import json
import math
import urllib.request
import numpy as np
import pandas as pd
import networkx as nx
from datetime import datetime, timezone

# Works when opened from the project root or notebooks/ directory.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "scripts/nyc_routes.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.nyc_routes import (RouteConfig, load_zones, plan, collect,
                               export_matrices, population_transition)
DATA = ROOT / "outputs/nyc_routes"


In [ ]:
zones = load_zones(DATA / "zones.csv")
assert len(zones) == 70, "Use the 70 aggregated groups, not the source census tracts."
groups_70 = pd.read_csv(DATA / "groups_70.csv")
assert groups_70.zone_id.tolist() == zones.zone_id.tolist()
print(f"{len(groups_70)} analysis groups; {zones.tract_count.sum()} source tracts already aggregated.")
print("Travel matrix: 70 × 70. Race counts: 70 rows.")
print(zones.groupby("borough").agg(groups=("zone_id", "size"), population=("TOTPOP", "sum")))
display(groups_70.head())


In [ ]:
from IPython.display import Image, display
display(Image(filename=str(DATA / "zones_map.png")))


## Verified 2020 Census race counts for every zone

The 2020 counts are available; no 2010 fallback is used. `scripts/prepare_zone_demographics.py` reads the local official New York P.L. 94-171 file, joins geographic headers and P1/P2 by LOGRECNO, and aggregates the fixed tract-to-zone crosswalk. Every tract and zone count was checked against the raw Census records, and tract totals were checked against separately published borough/county records.

The **70 × 7 race matrix** contains White alone, Black alone, American Indian/Alaska Native alone, Asian alone, Native Hawaiian/Other Pacific Islander alone, Some Other Race alone, and Two or More Races. These groups sum to each zone's population and include Hispanic and non-Hispanic persons. Race-alone groups exclude people reporting that race in combination with another.

Hispanic origin was a separate question in the 2020 Census. The separate **70 × 8 race/ethnicity matrix** contains Hispanic/Latino of any race plus the seven non-Hispanic race categories. Its groups also sum to population. A **70 × 2 × 7** array contains the full race-by-Hispanic-origin cross-tabulation. Do not add Hispanic totals to the seven all-origin race totals.

These are published April 1, 2020 counts for all ages, not estimates of today's population. Sources: [Census P1](https://data.census.gov/table/DECENNIALPL2020.P1) and [Census P2](https://data.census.gov/table/DECENNIALPL2020.P2). Recompute locally with `.venv/bin/python scripts/prepare_zone_demographics.py`.


In [ ]:
DEMOGRAPHICS_DIR = DATA / "demographics_2020"
demographics = np.load(DEMOGRAPHICS_DIR / "demographics_2020.npz", allow_pickle=False)
assert np.array_equal(demographics["zone_ids"], zones["zone_id"].to_numpy())
race_names = demographics["race_names"].tolist()
race_counts = demographics["race_counts"]              # 70 zones × 7 groups
race_proportions = demographics["race_proportions"]
race_ethnicity_counts = demographics["race_ethnicity_counts"]  # 70 × 8
race_ethnicity_proportions = demographics["race_ethnicity_proportions"]
population = demographics["population"].astype(float)
assert np.array_equal(race_counts.sum(axis=1), population)
assert np.array_equal(race_ethnicity_counts.sum(axis=1), population)
print(json.loads((DEMOGRAPHICS_DIR / "validation.json").read_text())["checks"])
display(pd.read_csv(DEMOGRAPHICS_DIR / "race_counts_2020.csv"))


## Distance-only sensitivity scenarios

For different groups, use

$$\widehat{t}_{ij}=a+60\,\frac{k\,d_{ij}}{v},$$

where $d_{ij}$ is great-circle distance in km between the group representatives, $k$ is an assumed detour factor, $v$ is an assumed effective travel speed in km/hour, and $a$ is access/wait time in minutes.

| Scenario | Speed (km/h) | Detour factor | Access/wait (min) |
|---|---:|---:|---:|
| Faster | 25 | 1.2 | 5 |
| Baseline | 18 | 1.4 | 10 |
| Slower | 12 | 1.6 | 15 |

**These are adjustable assumptions, not measured NYC speeds or confidence bounds.** The model is symmetric and assumes every group is reachable. It does not know the road/transit network, bridges/tunnels, ferry schedules, traffic, transfers, or time of day; water crossings and Staten Island are particular weaknesses. A zero diagonal is a local modeling convention; real within-group travel is not zero.

All three scenarios produce complete 70 × 70 matrices without credentials or downloads. By default, generalized cost values only time (`trip_cost_usd=0`); this does not assume transit fares are free. Supply a justified trip-cost assumption if using a monetary fare. Offline results are stored separately from API responses and marked `approximate`.


In [ ]:
from scripts.approximate_travel import Approximation, export as export_approximation

TRAVEL_TIME_SOURCE = "subway_hybrid"  # Also available: "subway_only", "offline", "api".
OFFLINE_SCENARIO = "baseline"  # "faster", "baseline", or "slower"
OFFLINE_DIR = DATA / "offline_approximation"
ECONOMICS = dict(beta=0.1, omega=15.0, c0=1.25)  # Illustrative, not calibrated.
SCENARIOS = {
    "faster": Approximation(effective_speed_kmh=25, detour_factor=1.2, access_wait_minutes=5),
    "baseline": Approximation(effective_speed_kmh=18, detour_factor=1.4, access_wait_minutes=10),
    "slower": Approximation(effective_speed_kmh=12, detour_factor=1.6, access_wait_minutes=15),
}
offline_report = export_approximation(zones, OFFLINE_DIR, scenarios=SCENARIOS,
                                      trip_cost_usd=0.0, **ECONOMICS)
print("Offline matrices:", offline_report["matrix_shape"], "— zero API calls")
display(pd.DataFrame(offline_report["scenarios"]).T)
offline_times = pd.read_csv(OFFLINE_DIR / OFFLINE_SCENARIO / "duration_minutes.csv", index_col=0)
display(offline_times.iloc[:5, :5])


## Subway network and schedules — local calculation, no Routes API

The downloaded [official MTA GTFS dataset](https://www.mta.info/developers) supplies stations, actual line shapes, trip stop sequences, service calendars and minimum transfer times. This notebook reads saved results; it never downloads data or invokes Google automatically.

**Trip estimate = approximate station access + scheduled waiting + scheduled train travel + transfers + approximate station egress.** The router considers multiple accessible stations and actual trip sequences, including express stops and staying aboard a train. Trips are directional and time-dependent. Line geometry is used for the map; connectivity and train times come from GTFS trips and stop times, not straight lines between arbitrary stations.

The saved example departs **October 20, 2026 at 08:00 America/New_York**, with a six-hour horizon. Its feed covers May 26–October 31, 2026. This is **2026 scheduled transportation with 2020 demographics**, not a historical 2020 transit reconstruction. The regular feed omits most temporary changes and is not real-time performance data.

Walking assumptions: 4.8 km/h, distance multiplier 1.3, at most 3 km for each access/egress leg, 2 minutes each for entry/exit, and at most 5 km for a direct walk. A buffered Census land mask rejects straight walking segments across open water; it does not provide a street network and may miss real bridge approaches. Only official GTFS transfers are added, with a 2-minute minimum assumption even when GTFS reports zero. Remaining aboard a train adds no transfer penalty.

The model covers **3,564 of 4,830 nonlocal pairs** (3,494 by scheduled rail and 70 by direct walking). The hybrid version fills **1,266 uncovered pairs** with the previous distance-only baseline, labeled `DISTANCE_FALLBACK`. Seven groups have no station within the modeled walking-access limits. Staten Island has SIR service but no subway connection to the other boroughs; ferries and buses are not included. `NO_PATH_IN_MODEL` does not mean no real trip exists.

The full matrix stays 70 × 70. `subway_only` leaves unavailable cells unresolved; `hybrid` provides a complete approximation. Journey components and station/line identities are available for inspection in `journeys.jsonl`. Generalized cost uses time only, with the same illustrative beta/omega/c0 settings; fares are not estimated.

Recompute locally from the saved dataset:

```bash
.venv/bin/python -m scripts.subway_travel --date 2026-10-20 --departure 08:00:00
```

Use `--download` explicitly to refresh the public MTA ZIP before a future run. A service date outside the feed validity raises an error. If economic parameters are changed, rebuild these saved matrices with matching parameters before comparing indices.


In [ ]:
SUBWAY_DIR = DATA / "subway_network/2026-10-20_080000"
subway_manifest = json.loads((SUBWAY_DIR / "manifest.json").read_text())
assert subway_manifest["groups"] == 70
print("Subway/walking coverage:", subway_manifest["network_covered_nonlocal_pairs"], "of 4830")
print("Distance-only fallback:", subway_manifest["distance_fallback_pairs"])
display(pd.read_csv(SUBWAY_DIR / "coverage_by_group.csv"))
display(Image(filename=str(SUBWAY_DIR / "subway_groups_map.png")))


In [ ]:
# Inspect one modeled trip; change these group IDs to compare other locations.
example_origin, example_destination = "MN04", "BK18"
with (SUBWAY_DIR / "journeys.jsonl").open() as stream:
    example_journey = next(json.loads(line) for line in stream
                           if (record := json.loads(line))["origin"] == example_origin
                           and record["destination"] == example_destination)
print(example_journey["mode"], round(example_journey["minutes"], 1), "minutes")
display(pd.DataFrame(example_journey["legs"]))


## Option B — Routes API dataset (collection still disabled)

The departure below is a **proposed Tuesday 8 a.m. New York snapshot**, not an approved collection time. Choose the actual time before approval and collection. Every pair uses that same departure time. A different departure time, mode set, or zone definition requires a separate cache. Beta, omega, c0, and price overrides can be changed offline without collecting routes again.

Google can return a transit route without a fare. Such pairs retain their travel-time data but have unknown cost/weight until a justified fare assumption is provided. No default fare is invented. See [Google's transit documentation](https://developers.google.com/maps/documentation/routes/transit-route).


In [ ]:
CONFIG = RouteConfig(departure_time="2026-10-20T08:00:00-04:00", modes=("TRANSIT",))
CACHE = DATA / "transit_2026-10-20_0800.sqlite"
MATRIX_DIR = DATA / "transit_2026-10-20_0800"
plan(zones, CONFIG)


In [ ]:
# This creates/reads the API dataset offline; it does not collect anything.
summary = export_matrices(zones, CONFIG, CACHE, MATRIX_DIR, **ECONOMICS)
print("API dataset:", summary["status_counts"])
api_matrices = np.load(MATRIX_DIR / "matrices.npz", allow_pickle=False)


## Disabled collection cell

Only after explicit approval: choose the modes and time, configure `GOOGLE_MAPS_API_KEY` locally, and deliberately enable the flag and a request cap. A cap of 1 supports an initial single-request check; 4,830 permits a full fresh one-mode run. No key is written to disk. There are no automatic retries. Each successful response is committed immediately; resuming skips saved successes. Failed or interrupted requests require an explicit `retry_errors=True` decision because they may already have been billed.

Run the export cell again after collection. Empty route results become `no_route` with cost infinity and weight zero. Pending requests, request errors, and unknown prices stay NaN. The 70 diagonal times/distances are set to zero by modeling convention, not observed within-zone travel.


In [ ]:
ALLOW_API = False  # Leave False until explicitly approved.
MAX_REQUESTS = 0   # Also requires a deliberate positive cap.
if ALLOW_API:
    collection_result = collect(zones, CONFIG, CACHE, allow_api=True,
                                max_requests=MAX_REQUESTS)
    print(collection_result)
else:
    print("Routes API disabled. No requests made.")


## Final WALK/TRANSIT cost and segregation matrices

The earlier sections show travel-time models. **The final cost matrix below supersedes their time-only social weights.** Only WALK and TRANSIT enter the minimum; the transit proxy is a labeled estimate within TRANSIT, not a third travel mode.

$$C_{ij}=\max\{c_0,\min(\omega t^{\mathrm{WALK}}_{ij},\;p^{\mathrm{TRANSIT}}_{ij}+\omega t^{\mathrm{TRANSIT}}_{ij})\},\quad F_{ij}=e^{-\beta C_{ij}}.$$

Times are converted from seconds to hours before multiplying by omega. Walking costs $0 out of pocket. The standard adult fare is $3 for a paid transit journey, with free SIR-only trips when neither endpoint is St George nor Tompkinsville. Normal transfers are treated as part of one fare; passes, fare caps, discounts, premium services and rider fare history are not modeled. Sources: [MTA fares](https://www.mta.info/fares-tolls/subway-bus) and [MTA tariff](https://www.mta.info/document/20581).

The computation retains walking and transit candidates independently. It selects by **fare plus time value**, not by fastest time. For example, at $15/hour, a 20-minute walk costs $5 while a 10-minute paid subway trip costs $5.50. It also retains separate zero-fare and paid transit candidates before comparing their costs. The local diagonal always has `C[i,i]=c0` and `F[i,i]=exp(-beta*c0)`.

Illustrative parameters remain beta=0.1 per dollar, omega=$15/hour, c0=$1.25 (five minutes at that omega). They are shared across all groups. If omega changes, c0 remains an independent fixed-dollar parameter unless explicitly changed as well.

Direct walking now has no arbitrary 5 km cutoff, but still requires a land-supported approximate connector. Transit retains the saved schedule's 3 km station access/egress cap and six-hour window. The complete `estimated` variant uses the previous baseline as a **hypothetical transit time with a $3 fare** only where scheduled transit candidates are absent. With the supplied parameters it selects 181 WALK, 3,365 scheduled TRANSIT, and 1,284 proxy TRANSIT journeys, plus 70 local cells. This is an estimated model, not fully observed routing. The `scheduled_candidates_only` variant has no proxy and leaves 894 cells unresolved; it does not compute a full-city index from missing data.

For race r, all 70 nodes, including the diagonal, enter the denominator:

$$P_{ij}=\frac{F_{ij}D(j)}{\sum_k F_{ik}D(k)},\qquad A^{(r)}=P\,\mathrm{diag}(p_r).$$

The code obtains the Perron eigenvalue and its **right** eigenvector, normalized so that the vector entries sum to one. The section scores are relative eigenvector weights under this model, not percentages of individual segregation. Seven exhaustive race groups and a separate eight-group race/ethnicity breakdown are exported. Rows and columns retain the same 70-group ordering. Formula checks compare the exported A matrices with the literal expression above.


## Four-category comparison and normalization

**White alone, Black alone, Hispanic/Latino (any race), and Asian alone** use the
verified 2020 counts for the same 70 sections. The three race categories include
Hispanic residents. These four categories overlap and do not form a population
partition; their proportions are never rescaled to sum to one.

For each category $r$, use the population-weighted citywide share
$$K_r=\frac{\sum_i D(i)\mathbb{P}_i(r)}{\sum_i D(i)},\qquad
S_r^{\mathrm{normalized}}=\frac{S_r-K_r}{1-K_r}.$$
$K$ is a fraction (not a percentage from 0 to 100). Apply this to PSSI. For Fryer, the requested population-share adjustment is
**$S_r-K_r$**, without dividing by $1-K_r$. The earlier $(S_r-K_r)/(1-K_r)$
calculation remains as a separately labeled rescaled comparison. Negative values are retained, and $K=1$ is undefined. This is the
requested population-share adjustment, not a simulated random-mixing correction.
The paper (pp. 24–25) subtracts a **simulated baseline SSI**, which is not generally
equal to $K$. That baseline is not computed here; $S-K$ is labeled explicitly.
Right eigenvectors still sum to one; this scalar adjustment is **not** applied
to their individual components.

The **Fryer majority-section SSI** follows the original notebook's $p_i\geq0.5$
rule. Use the same row-stochastic population-weighted interaction matrix
$P_{ij}=F_{ij}D(j)/\sum_k F_{ik}D(k)$ as in $A_r=P\operatorname{diag}(p_r)$.
Restrict $P$ to majority sections, retain its diagonal and original weights,
and average the components' spectral radii weighted by their section counts.
Do not renormalize the restricted rows: connections outside the selected sections
must still reduce the score. With no majority sections the notebook assigns zero;
that does not establish an absence of segregation among those residents.

This is a 70-section adaptation of [Echenique and Fryer's SSI](https://eml.berkeley.edu/~fechenique/published/echenique-fryer-qje.pdf).
Their residential application classifies Census blocks by majority and uses a
one-kilometer neighbor rule. Here the sections are much larger and interactions
come from modeled travel costs. Both metrics use our $P$ for comparison. The
normalization requested here uses **population share**, not the fraction of
majority sections, even for Fryer's section-based calculation.

Additional computations compare each category to all other residents:

- **Residential isolation:** $\sum_i [D(i)p_i/N_r]p_i$.
- **Network isolation:** $\sum_i [D(i)p_i/N_r](Pp)_i$; expected same-group exposure
  averaged over the group's residents. Both isolation scores also get the $K$ adjustment.
- **Dissimilarity:** $\tfrac12\sum_i|D(i)p_i/N_r-D(i)(1-p_i)/(N-N_r)|$.
  This already has its usual 0–1 scale; no extra adjustment is applied.

The same approximate WALK/TRANSIT costs, illustrative economic parameters and
labeled transit proxies remain in use. No Routes API calls are made.

The wide table shows every output column. Its column dictionary below defines each
heading, formula and unit. A standalone scrollable version is saved as
`cost_model/estimated/four_category_comparison.html`.


In [ ]:
from scripts.cost_matrices import build as build_cost_model, race_analysis
from scripts.segregation_metrics import select_four_categories, comparison_column_dictionary

COST_PARAMETERS = dict(beta=0.1, omega=15.0, c0=1.25, transit_fare=3.0)
USE_TRANSIT_PROXY = True  # False leaves unsupported pairs unresolved; no full-city index.
cost_report = build_cost_model(**COST_PARAMETERS)  # Local saved data only; no API calls.
COST_DIR = DATA / "cost_model" / ("estimated" if USE_TRANSIT_PROXY else "scheduled_candidates_only")
with np.load(COST_DIR / "cost_matrices.npz", allow_pickle=False) as saved:
    cost_matrices = {key: saved[key] for key in saved.files}
assert cost_matrices["zone_ids"].tolist() == zones.zone_id.tolist()
C = cost_matrices["C_usd"]
F = cost_matrices["F"]
T_seconds = cost_matrices["selected_seconds"]
chosen_mode = cost_matrices["chosen_mode"]
chosen_source = cost_matrices["chosen_source"]
assert C.shape == F.shape == (70, 70)
print(cost_report["parameters"])
print(cost_report["variants"])
display(pd.DataFrame(C, index=zones.zone_id, columns=zones.zone_id).iloc[:5, :5])

# Requested four categories, in this order. Hispanic is any race and overlaps.
comparison_names, comparison_counts, comparison_proportions = select_four_categories(demographics)
population = demographics["population"].astype(float)
selected_race = "black_alone"  # Any name in comparison_names can be inspected below.
race_index = comparison_names.tolist().index(selected_race)
p = comparison_proportions[:, race_index]
prop_matrix, pop_matrix = np.diag(p), np.diag(population)
ready = np.isfinite(C).all()
if ready:
    P, all_A, lambdas, right_vectors, residuals = race_analysis(
        cost_matrices["log_F"], population, comparison_proportions)
    A, index, v = all_A[race_index], lambdas[race_index], right_vectors[:, race_index]
    assert all_A.shape == (4, 70, 70)
    assert np.allclose(A, (F * (population * p)[None, :]) / (F @ population)[:, None])
    assert np.isclose(v.sum(), 1)
    assert np.allclose(A @ v, index * v)
    comparison = pd.read_csv(COST_DIR / "four_category_indices.csv")
    assert comparison.group.tolist() == comparison_names.tolist()
    assert np.allclose(comparison.population_share_K, comparison_counts.sum(axis=0) / population.sum())
    column_dictionary = comparison_column_dictionary()
    column_labels = dict(zip(column_dictionary.column, column_dictionary.label))
    # Full table: excess (score-K) and normalized (score-K)/(1-K) are separate.
    with pd.option_context("display.max_columns", None, "display.precision", 6):
        display(comparison.rename(columns=column_labels))
    print("Wide HTML table:", COST_DIR / "four_category_comparison.html")
else:
    print("Unresolved costs:", int(np.isnan(C).sum()), "— no full-city segregation index computed.")


In [ ]:
if ready:
    # Definitions for every column in the comparison, including formulas and units.
    with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
        display(column_dictionary)
    section_diagnostics = pd.read_csv(COST_DIR / "four_category_section_diagnostics.csv")
    # All 280 category/section records are saved; inspect the top 10 for a category.
    display(section_diagnostics[section_diagnostics.group == selected_race]
            .sort_values("right_eigenvector_score", ascending=False).head(10))
    print("Four 70 × 70 A matrices and vectors:", COST_DIR / "four_category_analysis.npz")
    print("Comparison:", COST_DIR / "four_category_indices.csv")
    print("Per-section data:", COST_DIR / "four_category_section_diagnostics.csv")


## Original analysis functions

The original demographic loaders, graph helpers, and PSSI functions are retained below for reference. They are not used to construct the new zones. Historical download and live-request example cells have been disabled. Original geographic assumptions and older 2010 field names remain in these reference functions; use the new 2020 zone table for this study.


In [ ]:
from types import TracebackType
# Hashmap for each state (numbers are for the URL)
#Is Mississippi the most segregated state for black people?
#Bama 2
#
states_map = {
    1 : "AL", 2 : "AK", 4 : "AZ", 5 : "AR", 6 : "CA", 8 : "CO", 9 : "CT",
    10 : "DE", 12 : "FL", 13 : "GA", 15: "HI", 16 : "ID", 17 : "IL",
    18 : "IN", 19 : "IA", 20 : "KS", 21 : "KY", 22 : "LA", 23 : "ME",
    24 : "MD", 25: "MA", 26 : "MI", 27 : "MN", 28 : "MS", 29 : "MO",
    30 : "MT", 31: "NE", 32 : "NV", 33 : "NH", 34 : "NJ", 35 : "NM",
    36 : "NY", 37 : "NC", 38 :"ND", 39 : "OH", 40 : "OK", 41 : "OR",
    42 : "PA", 44 : "RI", 45 : "SC", 46 : "SD", 47 : "TN", 48 : "TX",
    49 : "UT", 50 : "VT", 51 : "VA", 53 : "WA", 54 : "WV", 55 : "WI",
    56 : "WY"
}

states_map_2 = {v: k for k, v in states_map.items()}

states_array = [1, 2, 4, 5, 6, 8, 9, 10, 12, 13, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 44, 45, 46, 47, 48, 49, 50, 51, 53, 54, 55, 56]

# Demographics list from the 2010 census. Everything below this line is
# used to grab information about demographics from the .json file
r_original = {
    1 : "_BLACK",
    2 : "_WHITE",
    3 : "_ASIAN",
    4 : "HISP"
}

r_map = {}
for i in range(1, 4):
    string_arr = ['NH', 'H']
    result = [s + r_original[i] for s in string_arr]
    r_map[i] = result

r_map[4] = ['HISP']

# Initializes URL map in the COUNTY level (less nodes)
c_map = {}
for i in range(1, 57):
    index_str = str(i)
    url = "https://people.csail.mit.edu/ddeford/COUNTY/COUNTY_"
    if(i < 10):
        index_str = str(0) + index_str
    c_map[i] = url + index_str + ".json"

tract_map = {}
for i in range(1, 57):
    index_str = str(i)
    url = "https://people.csail.mit.edu/ddeford/BLOCK/BLOCK_"
    if(i < 10):
        index_str = str(0) + index_str
    tract_map[i] = url + index_str + ".json"
# Initializes URL map in the TRACT level (more nodes)
url_map = {}
for i in range(1, 57):
    index_str = str(i)
    url = "https://people.csail.mit.edu/ddeford/TRACT/TRACT_"
    if(i < 10):
        index_str = str(0) + index_str
    url_map[i] = url + index_str + ".json"

In [ ]:
# Function to sum the total population of a certain demographic within a census unit
# Input:
#   - data : data["nodes"][ID], where ID is the county or tract ID from the file
#   - r1   : demographic array
#            example: r_map[1] is the demographic array for the black population
#
# Output:
#   - integer value for total population of a certain demographic within the
#     given geographic unit
def sum_total(data, r1):
    result = 0
    for i in range(0, len(r1)):
        result += data[r1[i]]

    return result

In [ ]:
# Function to create an array of cartesian coordinates for each
# census unit, along with a proportion for the desired demographic
#
# Input:
#   - url : a url, which can be taken from url_map
#   - r1 : a demographic array, taken from r_map
#   - url2 : a county url, taken from c_map
#
# Output:
#   - an array structured as follows:
#     [x coordinates, y coordinates, proportion of demographic]
def load_array(url, r1, url2):
    response = urllib.request.urlopen(url)
    data = json.loads(response.read())


    node_data = data["nodes"]


    x_arr_1 = []
    y_arr_1 = []
    prop_arr_1 = []
    total_pop_arr_1 = []
    total_pop = 0

    for i in range(0, len(node_data)):
        x_arr_1.append(node_data[i]['C_X'])
        y_arr_1.append(node_data[i]['C_Y'])
        if(node_data[i]['TOTPOP'] == 0):
            prop_arr_1.append(0)
            total_pop_arr_1.append(0)
        else:
            total_r = sum_total(node_data[i], r1)
            total_pop = node_data[i]['TOTPOP']
            prop_arr_1.append(total_r/total_pop)
            total_pop_arr_1.append(total_pop)

    return [x_arr_1, y_arr_1, prop_arr_1, total_pop_arr_1]

In [ ]:
'''
I want to create a method that will load an array in the same format for a singular county.

INPUT:
url - Block url from which data is taken
county_num - The number of the county
r1 - Demographic array

OUTPUT:
an array structured as follows:
[x coordinates, y coordinates, proportion of demographic, total_pop]
'''


def load_within_county(url, county_num, r1):
    response = urllib.request.urlopen(url)
    data = json.loads(response.read())

    node_data = data["nodes"]
    prop_arr_1 = []
    total_pop_arr_1 = []

    x_arr_1 = []
    y_arr_1 = []

    for i in range(len(node_data)):
        if node_data[i]["COUNTYFP10"] == county_num:
            x_arr_1.append(node_data[i]['C_X'])
            y_arr_1.append(node_data[i]['C_Y'])
            if(node_data[i]['TOTPOP'] == 0):
                prop_arr_1.append(0)
                total_pop_arr_1.append(0)
            else:
                total_r = sum_total(node_data[i], r1)
                total_pop = node_data[i]['TOTPOP']
                prop_arr_1.append(total_r/total_pop)
                total_pop_arr_1.append(total_pop)
    return [x_arr_1, y_arr_1, prop_arr_1, total_pop_arr_1]

In [ ]:
def create_all_county_maps(r1, state_num):
    county_url = c_map[state_num]
    print(county_url)
    response = urllib.request.urlopen(county_url)
    data = json.loads(response.read())

    all_maps = []

    node_data = data["nodes"]

    for i in data["nodes"]:
        all_maps.append(load_within_county(tract_map[state_num], i["COUNTYFP10"], r1))
        print(i["NAME10"])

    return all_maps

In [ ]:
import math

def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate the great circle distance in kilometers between two points
    on the earth (specified in decimal degrees)
    """
    # Convert decimal degrees to radians
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])

    # Haversine formula
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    r = 6371  # Radius of earth in kilometers
    distance = r * c

    return distance


In [ ]:
def create_graph_from_2darray(arr):
    G = nx.Graph()
    for i in range(0, len(arr[0])):
        if arr[3][i] != 0:
          G.add_node(i, pos=(arr[0][i], arr[1][i]), prop=arr[2][i], pop=arr[3][i])
    for node,data in G.nodes(data=True):
        for node2,data2 in G.nodes(data=True):
            if haversine_distance(data['pos'][0], data['pos'][1], data2['pos'][0], data2['pos'][1]) <= 1:
                G.add_edge(node, node2)
    return G

In [ ]:
#For this matrix then you need to multiply by the proportion of all the connections.

def get_prob_matrix(adj_matrix):
    prob_matrix = np.zeros((adj_matrix.shape[0], adj_matrix.shape[0]))
    for i in range(adj_matrix.shape[0]):
        sum = 0
        for j in range(adj_matrix.shape[1]):
            sum += adj_matrix[i][j]
        for j in range(adj_matrix.shape[1]):
            prob_matrix[i][j] = (adj_matrix[i][j] / sum)
    return prob_matrix

def get_pop_density_matrix(adj_matrix, total_pops_array):
    prob_matrix = np.zeros((adj_matrix.shape[0], adj_matrix.shape[0]))
    for i in range(adj_matrix.shape[0]):
        sum = 0
        for j in range(adj_matrix.shape[1]):
            sum += (adj_matrix[i][j] * total_pops_array[j])
        for j in range(adj_matrix.shape[1]):
            prob_matrix[i][j] = ((adj_matrix[i][j]* total_pops_array[j]) / sum)
    return prob_matrix


def get_relevant_matrices(G):
    props = np.zeros((G.number_of_nodes(), G.number_of_nodes()))
    A = get_prob_matrix(nx.adjacency_matrix(G).toarray())
    val = 0
    for node,data in G.nodes(data=True):
        props[val][val] = data['prop']
        val = val + 1
    return A, props

def get_relevant_density_matrices(G):
    total_pops_array = np.zeros(G.number_of_nodes())
    x = 0
    for node,data in G.nodes(data=True):
        total_pops_array[x] = data['pop']
        x = x + 1
    props = np.zeros((G.number_of_nodes(), G.number_of_nodes()))
    A = get_pop_density_matrix(nx.adjacency_matrix(G).toarray(), total_pops_array)
    val = 0
    for node,data in G.nodes(data=True):
        props[val][val] = data['prop']
        val = val + 1
    return A, props

In [ ]:
#Individually feed unconnected subgraphs into this method
#FIX FOR SPARSE MATRICES
'''
prop matrix is a diagonal matrix with each proportion there

soc_matrix is the social matrix with the represeenting interactions between every two nodes
'''
def get_discrete_matrix(prop_matrix):
    discrete_matrix = np.zeros((prop_matrix.shape[0],prop_matrix.shape[0]))
    for i in range(prop_matrix.shape[0]):
        for j in range(prop_matrix.shape[1]):
            if(prop_matrix[i][j] < 0.5):
                discrete_matrix[i][j] = 0
            else:
                discrete_matrix[i][j] = 1
    return discrete_matrix


def Fryer_SSI(soc_matrix, prop_matrix):
    disc_prop_matrix = get_discrete_matrix(prop_matrix)
    alt_g = nx.Graph()
    nums = 0
    for i in range(len(disc_prop_matrix)):
        if disc_prop_matrix[i][i] == 1:
            alt_g.add_node(i)
            nums +=1
            for j in range(0,i):
                if disc_prop_matrix[j][j] == 1 and soc_matrix[i][j] > 0:
                    alt_g.add_edge(i,j)
    S = [alt_g.subgraph(c).copy() for c in nx.connected_components(alt_g)]
    total = 0
    for i in S:
      locs = []
      for a in i.nodes():
        locs.append(a)
      submatrix = soc_matrix[np.ix_(locs, locs)]
      total+= (np.real(max(np.linalg.eigvals(submatrix))) * len(locs))
    if nums == 0:
        return 0
    return (total / nums)

In [ ]:
import numpy as np

def PSSI(soc_matrix, prop_matrix, pop_matrix, return_vector=False):
    """
    Compute the population-weighted segregation index.

    soc_matrix: (n, n) array of nonnegative social weights f(i -> j).
    prop_matrix: length-n vector or diagonal matrix of race proportions.
    pop_matrix: length-n vector or diagonal matrix of populations.
    return_vector: also return the normalized right eigenvector if True.

    Nodes with no reachable population receive an all-zero row in A.
    """
    F = np.asarray(soc_matrix, dtype=float)
    if F.ndim != 2 or F.shape[0] != F.shape[1]:
        raise ValueError("soc_matrix must be square.")

    n = F.shape[0]

    def as_vector(values, name):
        values = np.asarray(values, dtype=float)
        if values.shape == (n, n):
            if not np.all(values == np.diag(np.diag(values))):
                raise ValueError(f"{name} must be diagonal.")
            values = np.diag(values)
        if values.shape != (n,):
            raise ValueError(f"{name} must be a length-n vector or diagonal matrix.")
        return values

    p = as_vector(prop_matrix, "prop_matrix")
    D = as_vector(pop_matrix, "pop_matrix")

    if not all(np.all(np.isfinite(x)) for x in (F, p, D)):
        raise ValueError("All inputs must contain finite values.")
    if np.any(F < 0) or np.any(D < 0):
        raise ValueError("Social weights and populations must be nonnegative.")
    if np.any((p < 0) | (p > 1)):
        raise ValueError("Race proportions must lie between 0 and 1.")

    # denominator[i] = sum_k f(i -> k) * D(k)
    denominator = F @ D

    # numerator[i, j] = f(i -> j) * D(j) * p(j)
    numerator = F * (D * p)[None, :]
    A = np.divide(
        numerator,
        denominator[:, None],
        out=np.zeros_like(F),
        where=denominator[:, None] > 0,
    )

    if not np.any(A):
        index = 0.0
        v = np.full(n, 1.0 / n) if n else np.array([])
    elif return_vector:
        eigenvalues, eigenvectors = np.linalg.eig(A)
        # A is nonnegative, so its spectral radius is a real eigenvalue.
        k = np.argmax(eigenvalues.real)
        index = float(eigenvalues[k].real)

        # Remove arbitrary eigenvector sign and numerical roundoff.
        v = np.abs(eigenvectors[:, k].real)
        v /= v.sum()
    else:
        index = float(np.linalg.eigvals(A).real.max())

    return (index, v) if return_vector else index

In [ ]:
# Reusable version of the original social-weight function, with offline replay support.
from scripts.social_weight import social_weight
# No live sample request: collection must go through the disabled opt-in cell above.
